In [6]:
import os
import json
import random
from datetime import datetime, timedelta

os.makedirs("data", exist_ok=True)

stores = ['Warszawa', 'Krakow', 'Gdansk', 'Wroclaw']
categories = ['elektronika', 'odziez', 'zywnosc', 'ksiazki']
start_time = datetime(2023, 10, 25, 8, 0, 0)

with open("data/transactions_10k.jsonl", "w") as f:
    for i in range(10000):
        tx_num = random.randint(1, 9999)
        tx_time = start_time + timedelta(seconds=random.randint(0, 10800))
        record = {
            'tx_id': f'TX{tx_num:04d}',
            'user_id': f'u{random.randint(1, 20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(stores),
            'category': random.choice(categories),
            'timestamp': tx_time.strftime("%Y-%m-%d %H:%M:%S")
        }
        f.write(json.dumps(record) + "\n")

print("Gotowe")

Gotowe


In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy\n")

df = spark.read.json("data/transactions_10k.jsonl")

from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print("\nZadanie 2.1 — Liczba transakcji i suma przychodów per sklep")
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()


print("\nZadanie 2.2 — Statystyki per kategoria")
from pyspark.sql.functions import min as _min, max as _max

kategorie_stat = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma"),
        _round(_min("amount"), 2).alias("min"),
        _round(_max("amount"), 2).alias("max")
    )
    .orderBy("category")
)
kategorie_stat.show()


print("\nZadanie 3.1 — Liczba transakcji per godzina (tumbling 1h)")
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.select(col("window.start").alias("od"), col("window.end").alias("do"), "liczba_tx", "suma_PLN").show(truncate=False)


print("\nZadanie 3.2 — Okna 30-minutowe per sklep")
okna_30m_sklep = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store")
)
okna_30m_sklep.select(col("window.start").alias("od"), col("window.end").alias("do"), "store", "liczba_tx", "suma_PLN").show(5, truncate=False)


print("\nZadanie 3.3 — W której godzinie sklep Kraków miał najwyższy przychód?")
from pyspark.sql.functions import desc

krakow_max_h = (
    df.filter(col("store") == "Krakow")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma_PLN"))
    .orderBy(desc("suma_PLN"))
)
krakow_max_h.select(col("window.start").alias("od"), col("window.end").alias("do"), "suma_PLN").show(1)


print("\nZadanie 4.1 — Okno 1h, krok 30 minut")
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(col("window.start").alias("od"), col("window.end").alias("do"), "liczba_tx", "suma_PLN")
    .orderBy("od")
)
sliding.show(5, truncate=False)


print("\nZadanie 4.2 — Porównaj tumbling vs sliding")
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)

sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)

print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# ODPOWIEDŹ: Sliding daje więcej wierszy, ponieważ okna przesuwno-nakładające się (sliding) sprawiają, że transakcje wpadają do wielu okien naraz. Przy oknie 
# 1h z krokiem 30 minut przedziały zachodzą na siebie, więc system generuje więcej unikalnych okien czasowych niż w sztywnych, nienakładających się oknach tumbling.


# Pytania kontrolne
# 1. Ile transakcji jest w oknie 09:00–10:00?
#    ODPOWIEDŹ: 3323 transakcje.
# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") agreguje dane z całego zbioru dla danego sklepu, a groupBy z oknem dzieli je dodatkowo na przedziały czasowe.
# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    ODPOWIEDŹ: 2 okna zawierają tę transakcję (okno 09:00-10:00 oraz okno 09:30-10:30).


print("\nPraca domowa")

print("\nZadanie 1")
gdansk_lowest_avg = (
    df.filter(col("store") == "Gdansk") 
    .groupBy(window("timestamp", "1 hour")) 
    .agg(_round(avg("amount"), 2).alias("srednia_kwota_PLN"))
    .orderBy("srednia_kwota_PLN") 
    .select(col("window.start").alias("od"), col("window.end").alias("do"), "srednia_kwota_PLN")
)
gdansk_lowest_avg.show(1) 

print("\nZadanie 2")
kategorie_0900_0930 = (
    df.filter((col("timestamp") >= "2023-10-25 09:00:00") & (col("timestamp") < "2023-10-25 09:30:00"))
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_transakcji"))
    .orderBy(desc("liczba_transakcji"))
)
kategorie_0900_0930.show()

print("\nZadanie 3")
szczyt_15min = (
    df.groupBy(window("timestamp", "15 minutes")) 
    .agg(count("tx_id").alias("liczba_transakcji"))
    .orderBy(desc("liczba_transakcji")) 
    .select(col("window.start").alias("od"), col("window.end").alias("do"), "liczba_transakcji")
)
szczyt_15min.show(1)

Spark 4.0.0-preview2 — gotowy


Zadanie 2.1 — Liczba transakcji i suma przychodów per sklep
+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdansk|     2414|6119930.49|    2535.18|
|  Krakow|     2426|6120798.22|     2523.0|
|Warszawa|     2591|6370319.88|    2458.63|
| Wroclaw|     2569|6446464.76|    2509.33|
+--------+---------+----------+-----------+


Zadanie 2.2 — Statystyki per kategoria
+-----------+----------+----+-------+
|   category|      suma| min|    max|
+-----------+----------+----+-------+
|elektronika|6429674.54| 6.3|4998.59|
|    ksiazki|6067301.24|8.47|4999.36|
|     odziez|6299320.05|5.75|4999.91|
|    zywnosc|6261217.52|8.37|4996.46|
+-----------+----------+----+-------+


Zadanie 3.1 — Liczba transakcji per godzina (tumbling 1h)
+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+----